In [1]:
import pandas as pd
import numpy as np

#***1***
<p>Generate 3,000 fictional UPI-style transactions in a notebook named 'upi_dates.ipynb' with seed 42: txn_id, txn_time as text 'DD/MM/YYYY HH:MM' between 01/01/2025 and 30/06/2025, amount_inr 10 to 5000, and category (Grocery, Fuel, Recharge, Food, Bills). Save 'upi_raw.csv', reload it and convert txn_time with `pd.to_datetime(format='%d/%m/%Y %H:%M')`, printing `dtypes`. Hint: Build times as a start date plus `pd.to_timedelta(minutes, unit='m')`, then `.dt.strftime`.</p>


In [2]:
df = pd.read_csv("applications_of_loan.csv")
df.head()

,age,monthly_income_inr,loan_amount_inr,tenure_months,employment,city_tier,credit_score,defaulted
0,59,44241,608158,24,Salaried,1,598,0
1,49,26745,677835,60,Salaried,1,660,0
2,35,41029,99067,36,Business,3,768,0
3,28,138395,162217,60,Self-employed,1,606,0
4,41,104186,468184,60,Self-employed,1,832,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   age                 1500 non-null   int64 
 1   monthly_income_inr  1500 non-null   int64 
 2   loan_amount_inr     1500 non-null   int64 
 3   tenure_months       1500 non-null   int64 
 4   employment          1500 non-null   object
 5   city_tier           1500 non-null   int64 
 6   credit_score        1500 non-null   int64 
 7   defaulted           1500 non-null   int64 
dtypes: int64(7), object(1)
memory usage: 93.9+ KB


In [4]:
df.shape

(1500, 8)

In [5]:
df.isnull().sum()

,0
age,0
monthly_income_inr,0
loan_amount_inr,0
tenure_months,0
employment,0
city_tier,0
credit_score,0
defaulted,0


In [6]:
df['defaulted'].value_counts()   #  0->defaulted loan 1->non defaulted loan

,count
defaulted,
0,1321
1,179


#***2***
<p>Extract `year`, `month`, `day`, `weekday` (day name), `quarter` and `hour` from txn_time with the `.dt` accessor. Create `is_weekend`, `is_month_end` and `time_slot` (Morning 5 to 11, Afternoon 12 to 16, Evening 17 to 21, Night otherwise) as new time-based features, and print the transaction count per `time_slot`. Constraint: Build `time_slot` with `.loc` masks, not `pd.cut`.</p>


In [7]:
df = pd.read_csv("transactions.csv")
df.head()

,txn_id,txn_time,amount_inr,payment_method,category
0,1,2025-09-26 12:36:44,3631,UPI,Entertainment
1,2,2025-07-06 06:41:18,7247,Net Banking,Bills
2,3,2025-09-27 23:57:30,4551,Net Banking,Shopping
3,4,2025-11-07 20:42:47,3976,Cash,Entertainment
4,5,2025-09-02 00:03:08,6059,UPI,Food


In [8]:
df['txn_time'] = pd.to_datetime(df['txn_time'])

df['year'] = df['txn_time'].dt.year
df['month'] = df['txn_time'].dt.month
df['day'] = df['txn_time'].dt.day

In [9]:
df['weekday'] = df['txn_time'].dt.day_name()
df['quarter'] = df['txn_time'].dt.quarter
df['hour'] = df['txn_time'].dt.hour

In [10]:
df['is_weekend'] = df['txn_time'].dt.dayofweek >= 5

df['is_month_end'] = df['txn_time'].dt.is_month_end

In [11]:
df['time_slot'] = 'Night'

In [12]:
df.loc[(df['hour'] >= 5) & (df['hour'] <= 11), 'time_slot'] = 'Morning'

df.loc[(df['hour'] >= 12) & (df['hour'] <= 16), 'time_slot'] = 'Afternoon'

df.loc[(df['hour'] >= 17) & (df['hour'] <= 21), 'time_slot'] = 'Evening'

In [13]:
df['time_slot'].value_counts()

,count
time_slot,
Morning,432
Night,428
Afternoon,322
Evening,318


#***3***
<p>Set txn_time as a sorted index, then use `resample` to build `daily_total` ('D'), `weekly_total` ('W') and `monthly_total` (month-end) sums of amount_inr. Assert the three grand totals are equal and save them as 'upi_daily.csv', 'upi_weekly.csv' and 'upi_monthly.csv'. Hint: Month-end is 'ME' in recent pandas and 'M' in older versions.</p>


In [14]:
df['txn_time'] = pd.to_datetime(df['txn_time'])

df = df.set_index('txn_time').sort_index()

In [15]:
daily_total = df['amount_inr'].resample('D').sum()

In [16]:
weekly_total = df['amount_inr'].resample('W').sum()

In [17]:
monthly_total = df['amount_inr'].resample('ME').sum()

In [18]:
assert daily_total.sum() == weekly_total.sum() == monthly_total.sum()

In [19]:
daily_total.to_csv('upi_daily.csv')

weekly_total.to_csv('upi_weekly.csv')

monthly_total.to_csv('upi_monthly.csv')

#***4***
<p>Treat txn_time as UTC server time: apply `dt.tz_localize('UTC')` then `dt.tz_convert('Asia/Kolkata')` to make `txn_time_ist`. Print the first row before and after (the shift must be 5 hours 30 minutes), compare the busiest hour before and after conversion, and save 'upi_clean.csv'. Run Restart & Run All and push to 's09_upi_dates' in your GitHub repo 'ds_analysis_portfolio'.</p>


In [20]:
df = df.reset_index()

print(df['txn_time'].iloc[0])

2025-01-01 12:06:25


In [21]:
df['txn_time_ist'] = df['txn_time'].dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata')

In [22]:
df['txn_time_ist'].iloc[0]

Timestamp('2025-01-01 17:36:25+0530', tz='Asia/Kolkata')

In [23]:
df['txn_time'].dt.hour.value_counts().head(1)

,count
txn_time,
6,77


In [24]:
df['txn_time_ist'].dt.hour.value_counts().head(1)

,count
txn_time_ist,
1,81


In [25]:
df.to_csv('upi_clean.csv', index=False)